# Lab 01-02 — Token-aware splitting: a token budget is not a character budget

**Track 01 · Chunking** — the second decision in every RAG pipeline is *what unit do I budget chunks in?* This lab runs the SAME budget number (300) through two splitters where the number means different units — and then measures both outputs in real token space with tiktoken.

This notebook is **self-contained**: it imports LangChain splitters and tiktoken directly — no repo component library. Every block of the pipeline is built right here: the Gutenberg loader (a plain file read plus a marker strip), the token splitter, and the recursive character splitter all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

```
raw text -> inline Gutenberg stripper -> TokenTextSplitter (300 TOKENS)     -> tiktoken measurement
                                       -> RecursiveCharacterTextSplitter (300 chars) -> tiktoken measurement
Data  : Data/corpus/gutenberg/ (pride-and-prejudice, moby-dick, a-tale-of-two-cities)
Budget: 300 tokens vs 300 characters, overlap 30 (same number, different unit)
Measure: tiktoken cl100k_base — the same encoder the repo's token splitter uses
```

**TOKEN splitting** cuts on token boundaries, so every chunk is guaranteed to stay inside the token budget the LLM actually bills against.

**CHARACTER splitting** cuts on character counts, which say nothing about tokens: a 300-character chunk can cost anywhere from ~20 to 300+ tokens depending on how token-dense the text is (markdown markup, code and symbols tokenize far heavier than plain prose).

Why it matters: context windows are priced in tokens. A character-budgeted pipeline silently overruns the budget whenever the text is token-dense, and only a token-aware splitter can guarantee your chunks fit the window you actually pay for.


## Setup

One prerequisite must hold before this notebook will run:

- **gutenberg corpus on disk** — `Data/corpus/gutenberg/` with `pride-and-prejudice.txt`, `moby-dick.txt`, `a-tale-of-two-cities.txt` (public-domain novels, already fetched by the repo's manifest-verified fetchers).

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-text-splitters`, and `tiktoken`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q langchain-core langchain-text-splitters tiktoken


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import statistics
from pathlib import Path

# LangChain splitters + tiktoken — the only libraries this notebook needs.
# Nothing is imported from the repo's src/ component library.
import tiktoken  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_text_splitters import RecursiveCharacterTextSplitter  # noqa: E402
from langchain_text_splitters import TokenTextSplitter  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

The whole comparison hangs on one trick: the SAME budget number `300` enforced in two different units. `TOKEN_CHUNK_SIZE`/`TOKEN_CHUNK_OVERLAP` budget chunks in TOKENS — the unit the LLM bills; `CHAR_CHUNK_SIZE`/`CHAR_CHUNK_OVERLAP` budget the same number in CHARACTERS. `DOC_PATHS` is the corpus (three public-domain Gutenberg novels, license boilerplate stripped before splitting) and `ENCODER_NAME` picks the tiktoken encoder — `cl100k_base`, the gpt-4 tokenizer, the same encoder the repo's token splitter measures with.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the comparison
# --------------------------------------------------------------------------
TOKEN_CHUNK_SIZE = 300            # budget in TOKENS — the unit the LLM bills
TOKEN_CHUNK_OVERLAP = 30
CHAR_CHUNK_SIZE = 300             # same number, different unit: CHARACTERS
CHAR_CHUNK_OVERLAP = 30
DOC_PATHS = [
    Path("Data/corpus/gutenberg/pride-and-prejudice.txt"),
    Path("Data/corpus/gutenberg/moby-dick.txt"),
    Path("Data/corpus/gutenberg/a-tale-of-two-cities.txt"),
]

# cl100k_base = the gpt-4 tokenizer; same encoder token_splitter.py measures
# with. Kept as a constant so the lab can be re-run under another encoding.
ENCODER_NAME = "cl100k_base"


## 2. Load — Gutenberg books, boilerplate stripped inline

The repo's `GutenbergLoader` strips everything between the `*** START OF THE PROJECT GUTENBERG EBOOK` and `*** END OF THE PROJECT GUTENBERG EBOOK` markers and sets `metadata["source"]` to the book path. The inline version below does exactly the same two things: read the file as UTF-8 text, slice out the book between the markers, and wrap it as one `Document` per book.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — Gutenberg books via the inline boilerplate stripper
# --------------------------------------------------------------------------
START_MARKER = "*** START OF THE PROJECT GUTENBERG EBOOK"
END_MARKER = "*** END OF THE PROJECT GUTENBERG EBOOK"


def strip_gutenberg_boilerplate(text: str) -> str:
    """Slice from just after the START marker to just before the END marker.

    If either marker is missing the text is returned unchanged (defensive:
    some mirrors drop the footer), and the result is stripped of leading/
    trailing whitespace — same semantics as the repo's GutenbergLoader.
    """
    start = text.find(START_MARKER)
    end = text.find(END_MARKER)
    if start == -1 or end == -1 or end < start:
        return text.strip()
    return text[start + len(START_MARKER):end].strip()


def load_documents(paths: list[Path]) -> list[Document]:
    """Load each Gutenberg book as a Document tagged with its source path."""
    docs: list[Document] = []
    for path in paths:
        text = strip_gutenberg_boilerplate(path.read_text(encoding="utf-8"))
        docs.append(Document(page_content=text, metadata={"source": str(path)}))
    return docs


## 3. Measure — every chunk in real token space

`token_counts` measures each chunk's real token consumption (not its char count) with tiktoken. `chunk_stats` summarizes a chunk population in token space — count, average, min, max, and the population std dev (the spread). `print_stats_row` renders one row of the comparison table. These are the same numbers the repo's token splitter lab reports.


In [ ]:
# --------------------------------------------------------------------------
# 3. Token-count — measure every chunk in real token space
# --------------------------------------------------------------------------
def token_counts(chunks: list[Document], enc: tiktoken.Encoding) -> list[int]:
    """Measure each chunk's real token consumption (not its char count)."""
    return [len(enc.encode(chunk.page_content)) for chunk in chunks]


def chunk_stats(counts: list[int]) -> dict[str, float]:
    """Summarize a chunk population in token space: count/avg/min/max/std."""
    n = len(counts)
    return {
        "chunks": n,
        "avg": sum(counts) / n,
        "min": min(counts),
        "max": max(counts),
        "std": statistics.pstdev(counts),  # full population of chunks
    }


def preview(text: str, limit: int = 200) -> str:
    """Truncate a chunk's content for printing."""
    return text[:limit] + ("..." if len(text) > limit else "")


def print_stats_row(label: str, counts: list[int], note: str) -> None:
    """Print one row of the comparison table, all numbers in token space."""
    s = chunk_stats(counts)
    print(
        f"{label:<38} {s['chunks']:>6} {s['avg']:>7.1f} "
        f"{s['min']:>4.0f} {s['max']:>4.0f} {s['std']:>7.1f}   {note}"
    )


## 4. Experiment — same budget through both splitters, then escalate

`run_experiment` runs the comparison in three acts. First it splits the same documents with `TokenTextSplitter(300 tokens)` and `RecursiveCharacterTextSplitter(300 chars)` — the lab's `TokenSplitter` and `DocumentProcessor` wrappers, used directly — and measures both outputs with tiktoken, so every reported number is in real token space. Then the escalation: the repo's default character budget is `chunk_size=1000` CHARACTERS, so the same splitter is re-run at that default to show the token cost silently climbing — nothing in the config mentions tokens, yet the chunks consume most of a 300-token budget. Finally the side-by-side inspection of the first chunk from each splitter.


In [ ]:
# --------------------------------------------------------------------------
# 4. Experiment — same documents through each splitter, then compare in tokens
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    enc = tiktoken.get_encoding(ENCODER_NAME)
    token_splitter = TokenTextSplitter(
        chunk_size=TOKEN_CHUNK_SIZE, chunk_overlap=TOKEN_CHUNK_OVERLAP
    )
    char_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHAR_CHUNK_SIZE, chunk_overlap=CHAR_CHUNK_OVERLAP
    )

    docs = load_documents(DOC_PATHS)
    token_chunks = token_splitter.split_documents(docs)
    char_chunks = char_splitter.split_documents(docs)
    token_tokens = token_counts(token_chunks, enc)
    char_tokens = token_counts(char_chunks, enc)

    # --- Escalation: the repo default char budget, in token terms ----------
    # DocumentProcessor's default is chunk_size=1000 CHARACTERS. Raise the
    # character budget and watch the token cost climb.
    default_char_splitter = RecursiveCharacterTextSplitter()  # repo default: 1000 chars
    default_char_chunks = default_char_splitter.split_documents(docs)
    default_char_tokens = token_counts(default_char_chunks, enc)
    default_stats = chunk_stats(default_char_tokens)

    return {
        "enc": enc,
        "docs": docs,
        "token_chunks": token_chunks,
        "char_chunks": char_chunks,
        "token_tokens": token_tokens,
        "char_tokens": char_tokens,
        "default_stats": default_stats,
        "spread": max(char_tokens) / min(char_tokens),
    }


## 5. Demo — print the artifact

`print_demo(exp)` prints the artifact from four angles: the loaded books; the chunk-count line for each splitter; the tiktoken-measured comparison table with the spread line; the escalation at the 1000-char default budget (with the % of the 300-token budget the max chunk silently consumed); the side-by-side first chunks; and the takeaway. All numbers are tiktoken token counts — the only unit an LLM context window understands.


In [ ]:
# --------------------------------------------------------------------------
# 5. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    enc = exp["enc"]
    docs = exp["docs"]
    token_chunks = exp["token_chunks"]
    char_chunks = exp["char_chunks"]
    token_tokens = exp["token_tokens"]
    char_tokens = exp["char_tokens"]
    default_stats = exp["default_stats"]
    spread = exp["spread"]

    print(f"Encoding: {ENCODER_NAME} (gpt-4 tokenizer)")
    print(
        f"Budget: {TOKEN_CHUNK_SIZE} tokens vs {CHAR_CHUNK_SIZE} characters, "
        f"overlap {TOKEN_CHUNK_OVERLAP}\n"
    )

    print(f"Loaded {len(docs)} document(s): {[p.name for p in DOC_PATHS]}")
    for doc in docs:
        print(f"  {Path(doc.metadata['source']).name}: {len(doc.page_content):,} chars")

    print(
        f"TokenSplitter ({TOKEN_CHUNK_SIZE} tokens): {len(token_chunks)} chunk(s); "
        f"DocumentProcessor ({CHAR_CHUNK_SIZE} chars): {len(char_chunks)} chunk(s)\n"
    )

    # Same nominal budget "300" — the only difference is the unit it is
    # enforced in. All numbers below are tiktoken token counts.
    print("Chunk count and token consumption, measured with tiktoken:")
    print(f"{'splitter':<38} {'chunks':>6} {'avg':>7} {'min':>4} {'max':>4} {'std dev':>7}")
    print("-" * 88)
    print_stats_row(
        f"TokenSplitter ({TOKEN_CHUNK_SIZE} tokens)",
        token_tokens,
        "bounded: every chunk <= budget",
    )
    print_stats_row(
        f"DocumentProcessor ({CHAR_CHUNK_SIZE} chars)",
        char_tokens,
        "no token ceiling: wide spread",
    )
    print("-" * 88)
    print(
        f"Token chunks: {min(token_tokens)}-{max(token_tokens)} tokens — uniform, "
        f"bounded at the {TOKEN_CHUNK_SIZE}-token budget (the few tokens over "
        f"300 are each document's final-chunk remainder)."
    )
    print(
        f"Char chunks: {min(char_tokens)}-{max(char_tokens)} tokens — a "
        f"{spread:.1f}x spread for the same '300' budget."
    )

    # ----------------------------------------------------------------------
    # 5. Escalation — the repo default char budget, in token terms
    # ----------------------------------------------------------------------
    s = default_stats
    print("\nEscalation — DocumentProcessor at its default budget (1000 chars):")
    print(
        f"  {s['chunks']:.0f} chunk(s), avg {s['avg']:.1f}, "
        f"min {s['min']:.0f}, max {s['max']:.0f} tokens per chunk"
    )
    print(
        f"  -> a '1000-character' budget silently consumed up to "
        f"{s['max']:.0f} tokens per chunk, ~{100 * s['max'] / TOKEN_CHUNK_SIZE:.0f}% "
        f"of the {TOKEN_CHUNK_SIZE}-token budget."
    )

    # ----------------------------------------------------------------------
    # 6. Inspect — one real chunk from each splitter, side by side
    # ----------------------------------------------------------------------
    print("\nSide by side — first chunk of each splitter:")
    for label, chunk, tokens in [
        ("TokenSplitter", token_chunks[0], token_tokens[0]),
        ("DocumentProcessor", char_chunks[0], char_tokens[0]),
    ]:
        print(f"  {label}: {tokens} tokens, {len(chunk.page_content)} chars")
        print(f"    {preview(chunk.page_content)!r}")

    # ----------------------------------------------------------------------
    # 7. Takeaway — token budget vs character budget
    # ----------------------------------------------------------------------
    print("\nTakeaway: a character budget is not a token budget.")
    print(
        f"- TokenSplitter({TOKEN_CHUNK_SIZE} tokens): every chunk is bounded at "
        f"the {TOKEN_CHUNK_SIZE}-token budget (measured "
        f"{min(token_tokens)}-{max(token_tokens)}); the budget is enforced in "
        f"the unit the LLM bills."
    )
    print(
        f"- DocumentProcessor({CHAR_CHUNK_SIZE} chars): the same '300' produced "
        f"chunks of {min(char_tokens)}-{max(char_tokens)} tokens ({spread:.1f}x spread). "
        f"Character count never determines token count — markdown, code and "
        f"symbols tokenize far heavier than prose — so no character budget "
        f"sets a token ceiling."
    )
    print(
        f"  (On these three novels the same nominal '300' produced "
        f"{len(char_tokens):,} char-budgeted chunks of "
        f"{min(char_tokens)}-{max(char_tokens)} tokens — a {spread:.0f}x spread, "
        f"driven by short fragments (chapter headings, whitespace runs) at the "
        f"low end. Plain prose runs ~5 chars/token, so a 300-char chunk lands "
        f"around {sum(char_tokens) // len(char_tokens)} tokens on average — well "
        f"under the 300-token budget — yet the character budget still sets no "
        f"token ceiling: at the repo's default 1000-char budget the max reached "
        f"{s['max']:.0f} tokens, ~{100 * s['max'] / TOKEN_CHUNK_SIZE:.0f}% of the "
        f"{TOKEN_CHUNK_SIZE}-token budget.)"
    )
    print(
        "- A token-aware splitter is the only one that can guarantee your "
        "chunks fit the context window you actually pay for."
    )


## 6. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: both splitters produced chunks; the token splitter's chunks are bounded at the budget (final-chunk remainders may nudge a few tokens over, but never beyond budget + overlap); the character splitter shows the wide token spread the lab is about; and the escalation is real — the 1000-char default budget consumes more tokens per chunk than the 300-char budget, and its max chunk blows past the 300-token budget. Every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 6. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    token_tokens = exp["token_tokens"]
    char_tokens = exp["char_tokens"]
    default_stats = exp["default_stats"]
    spread = exp["spread"]

    checks.append(("token splitter produced chunks", len(token_tokens) > 0))
    checks.append(("character splitter produced chunks", len(char_tokens) > 0))
    checks.append((
        f"token chunks bounded at the {TOKEN_CHUNK_SIZE}-token budget "
        f"(max {max(token_tokens)} <= budget + overlap)",
        max(token_tokens) <= TOKEN_CHUNK_SIZE + TOKEN_CHUNK_OVERLAP,
    ))
    checks.append((
        "char chunks show a wide token spread for the same '300' "
        f"(max/min = {spread:.1f}x > 2x)",
        spread > 2.0,
    ))
    checks.append((
        "escalation: 1000-char default budget consumes more tokens per chunk "
        f"than the 300-char budget ({default_stats['max']:.0f} vs {max(char_tokens)})",
        default_stats["max"] > max(char_tokens),
    ))
    checks.append((
        f"escalation: 1000-char max chunk blows past the {TOKEN_CHUNK_SIZE}-token "
        f"budget ({default_stats['max']:.0f} > {TOKEN_CHUNK_SIZE})",
        default_stats["max"] > TOKEN_CHUNK_SIZE,
    ))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few seconds: three plain-text reads, two splits, and tiktoken counts over ~2MB of public-domain prose — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The same nominal budget `300` in two units, measured in real token space: the token splitter's chunks are uniform and bounded; the character splitter's chunks have a wide token spread — and at the repo's default 1000-char budget the max chunk silently consumes ~400% of a 300-token budget.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the gutenberg files are intact.


In [ ]:
verify_gate(exp)
